# **Mecanismos de Atención**

Los mecanismos de atención se han convertido en uno de los conceptos más influyentes en el Deep Learning moderno, especialmente en el Procesamiento de Lenguaje Natural (NLP).

### **Concepto de Query, Keys y Values**
La atención se puede conceptualizar como una consulta en una base de datos asociativa:
* **Query (Consulta, $q$)**: Es el vector que representa lo que estamos buscando en el paso actual (por ejemplo, la palabra que se está decodificando).
* **Keys (Claves, $k_i$)**: Son los vectores que representan los elementos de la base de datos que indexan la información disponible (por ejemplo, las palabras de origen codificadas).
* **Values (Valores, $v_i$)**: Son los vectores que contienen el contenido real asociado a cada clave.

### **Ecuaciones de Atención de Producto Punto (Dot-Product Attention)**
Para calcular el foco de atención, medimos la similitud del producto punto entre la consulta $q$ y cada una de las claves $k_i$:
1. **Puntuaciones de Similitud (Scores)**: Mide el grado de alineación.
   $$\text{score}(q, k_i) = q \cdot k_i$$
2. **Pesos de Atención (Softmax, $\alpha_i$)**: Convierte las puntuaciones en una distribución probabilística de suma igual a 1.
   $$\alpha_i = \frac{\exp(\text{score}(q, k_i))}{\sum_{j} \exp(\text{score}(q, k_j))}$$
3. **Vector de Contexto ($c$)**: Es la combinación lineal ponderada de todos los valores $v_i$ según los pesos de atención $\alpha_i$.
   $$c = \sum_{i} \alpha_i v_i$$


# **1. Simulación del Mecanismo de Atención (Producto Punto)**

A continuación, realizaremos una simulación manual en PyTorch. Queremos simular la traducción de una palabra en la mente de la red (nuestra Query) basándonos en la representación de una oración origen de 6 palabras (nuestras Keys y Values).


In [11]:
import torch
import torch.nn.functional as F

# Query: Vector de la consulta (representa la palabra a traducir, ej. "sleeps")
vector_consulta = torch.tensor([1.8, 1.0])

# Vocabulario de la frase de origen (Keys y Values)
palabras_origen = ["El", "gato", "duerme", "en", "la", "cama"]

# Vectores clave (keys) asociados a cada palabra de origen
vectores_clave = torch.tensor([
    [0.1, 0.3],  # El
    [0.9, 1.1],  # gato
    [1.8, 1.0],  # duerme
    [0.2, 0.1],  # en
    [0.1, 0.2],  # la
    [0.3, 0.4]   # cama
])

# Valores (values) asociados a cada palabra de origen (en este caso, idénticos a las claves)
vectores_valor = vectores_clave.clone()


In [12]:
# Calcular puntuaciones de atención (producto punto entre claves y consulta)
puntuaciones = torch.matmul(vectores_clave, vector_consulta)

# Aplicar Softmax para obtener la distribución probabilística (pesos de atención)
pesos_atencion = F.softmax(puntuaciones, dim=0)
print("Pesos de atención por palabra calculados:")
print(pesos_atencion)


Pesos de atención por palabra calculados:
tensor([0.0176, 0.1653, 0.7560, 0.0173, 0.0159, 0.0279])


In [13]:
# Calcular el vector de contexto (suma ponderada de los valores por los pesos de atención)
vector_contexto = torch.sum(pesos_atencion.unsqueeze(1) * vectores_valor, dim=0)
print(f"Vector de contexto resultante: {vector_contexto.tolist()}\n")

# Mostrar la alineación de atención para cada palabra
for palabra, peso in zip(palabras_origen, pesos_atencion):
    print(f"{palabra:7s} → peso de atención asignado: {peso.item():.4f}")


Vector de contexto resultante: [1.5247595310211182, 0.9592127203941345]

El      → peso de atención asignado: 0.0176
gato    → peso de atención asignado: 0.1653
duerme  → peso de atención asignado: 0.7560
en      → peso de atención asignado: 0.0173
la      → peso de atención asignado: 0.0159
cama    → peso de atención asignado: 0.0279


# **2. Traducción Automática Seq2Seq con Mecanismo de Atención**

La traducción automática ha sido revolucionada por el uso de mecanismos de atención. Los modelos Seq2Seq tradicionales (Encoder-Decoder basados en RNN o LSTM) colapsaban toda la información de la frase de entrada en un único vector de contexto estático, limitando su rendimiento en secuencias largas.

Al añadir **atención**, el decodificador es capaz de examinar dinámicamente las salidas del codificador para cada paso de tiempo, enfocándose en las palabras relevantes en cada momento de la traducción.


In [14]:
# Instalación silenciosa de dependencias si fuesen necesarias
!pip install -q tqdm


In [15]:
# Importaciones unificadas al inicio del experimento
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F


In [16]:
# Diagnóstico de GPU/CUDA
esta_en_colab = 'google.colab' in sys.modules
print(f"¿Entorno de ejecución Google Colab?: {esta_en_colab}")

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo activo: {dispositivo}")

if torch.cuda.is_available():
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
else:
    print("No se detectó GPU CUDA. El entrenamiento se realizará en la CPU.")


¿Entorno de ejecución Google Colab?: False
Dispositivo de cómputo activo: cpu
No se detectó GPU CUDA. El entrenamiento se realizará en la CPU.


In [17]:
# 1. Corpus de entrenamiento Inglés -> Francés
oraciones_ingles = [
    "I love deep learning",
    "Machine translation is fascinating",
    "Attention mechanisms improve models"
]
oraciones_frances = [
    "<sos> J'adore l'apprentissage profond <eos>",
    "<sos> La traduction automatique est fascinante <eos>",
    "<sos> Les mécanismes d'attention améliorent les modèles <eos>"
]

# 2. Construcción de vocabulario
def crear_vocabulario(oraciones):
    """Genera una asociación entre palabras y sus correspondientes índices enteros."""
    vocabulario = {"<pad>": 0, "<unk>": 1}
    for oracion in oraciones:
        for palabra in oracion.lower().split():
            if palabra not in vocabulario:
                vocabulario[palabra] = len(vocabulario)
    return vocabulario

vocabulario_entrada = crear_vocabulario(oraciones_ingles)
vocabulario_salida = crear_vocabulario(oraciones_frances)
vocab_salida_inverso = {idx: palabra for palabra, idx in vocabulario_salida.items()}

# 3. Codificación de oraciones a tensores
def codificar_oracion(oracion, vocabulario, largo_maximo):
    """Convierte una oración de texto a una lista de índices con padding."""
    palabras = oracion.lower().split()
    indices = [vocabulario.get(palabra, vocabulario["<unk>"]) for palabra in palabras]
    # Agregar padding
    indices += [vocabulario["<pad>"]] * (largo_maximo - len(indices))
    return indices[:largo_maximo]

largo_max_entrada = max(len(s.split()) for s in oraciones_ingles)
largo_max_salida = max(len(s.split()) for s in oraciones_frances)

tensor_entrada = torch.tensor([codificar_oracion(s, vocabulario_entrada, largo_max_entrada) for s in oraciones_ingles])
tensor_salida = torch.tensor([codificar_oracion(s, vocabulario_salida, largo_max_salida) for s in oraciones_frances])

print(f"Largo máximo entrada (inglés): {largo_max_entrada}")
print(f"Largo máximo salida (francés): {largo_max_salida}")


Largo máximo entrada (inglés): 4
Largo máximo salida (francés): 8


In [18]:
dim_incrustacion = 256
tam_oculto = 512

class Codificador(nn.Module):
    """Codificador recurrente LSTM."""
    def __init__(self, tam_vocabulario, dim_incrustacion, tam_oculto):
        super(Codificador, self).__init__()
        self.incrustacion = nn.Embedding(tam_vocabulario, dim_incrustacion)
        self.lstm = nn.LSTM(dim_incrustacion, tam_oculto, batch_first=True)

    def forward(self, x):
        incrustada = self.incrustacion(x)
        salidas, (h, c) = self.lstm(incrustada)
        return salidas, (h, c)


class DecodificadorConAtencion(nn.Module):
    """Decodificador LSTM con mecanismo de atención por producto punto."""
    def __init__(self, tam_vocabulario, dim_incrustacion, tam_oculto):
        super(DecodificadorConAtencion, self).__init__()
        self.incrustacion = nn.Embedding(tam_vocabulario, dim_incrustacion)
        self.lstm = nn.LSTM(dim_incrustacion, tam_oculto, batch_first=True)
        self.lineal_atencion = nn.Linear(tam_oculto * 2, tam_oculto)
        self.capa_salida = nn.Linear(tam_oculto, tam_vocabulario)

    def forward(self, x, h, c, salidas_codificador):
        # x: (lote, 1), salidas_codificador: (lote, largo_entrada, tam_oculto)
        incrustada = self.incrustacion(x)  # (lote, 1, dim_incrustacion)
        salida_lstm, (h_n, c_n) = self.lstm(incrustada, (h, c))  # (lote, 1, tam_oculto)

        # 1. Puntuación de producto punto (Dot-product similarity)
        scores = torch.bmm(salida_lstm, salidas_codificador.transpose(1, 2))  # (lote, 1, largo_entrada)
        
        # 2. Pesos de atención (Softmax)
        pesos_atencion = F.softmax(scores, dim=-1)  # (lote, 1, largo_entrada)

        # 3. Vector de contexto (Suma ponderada de las claves)
        contexto = torch.bmm(pesos_atencion, salidas_codificador)  # (lote, 1, tam_oculto)

        # 4. Concatenación de salida LSTM con el vector de contexto
        combinado = torch.cat((salida_lstm, contexto), dim=-1)  # (lote, 1, tam_oculto * 2)
        salida_atencion = torch.tanh(self.lineal_atencion(combinado))  # (lote, 1, tam_oculto)

        # 5. Capa final de proyección sobre el vocabulario de destino
        salida_final = self.capa_salida(salida_atencion.squeeze(1))  # (lote, tam_vocabulario)
        return salida_final, h_n, c_n


# Inicializar el Codificador y el Decodificador con Atención
codificador = Codificador(len(vocabulario_entrada), dim_incrustacion, tam_oculto).to(dispositivo)
decodificador = DecodificadorConAtencion(len(vocabulario_salida), dim_incrustacion, tam_oculto).to(dispositivo)

def traducir_oracion(oracion):
    """Traduce una oración del inglés al francés usando decodificación codiciosa (greedy)."""
    codificador.eval()
    decodificador.eval()
    
    with torch.no_grad():
        indices_entrada = torch.tensor(
            codificar_oracion(oracion, vocabulario_entrada, largo_max_entrada)
        ).unsqueeze(0).to(dispositivo)
        
        salidas_codificador, (h, c) = codificador(indices_entrada)
        
        # Entrada inicial del decodificador: <sos>
        entrada_decodificador = torch.tensor([[vocabulario_salida["<sos>"]]]).to(dispositivo)
        palabras_traducidas = []
        
        for _ in range(largo_max_salida):
            salida, h, c = decodificador(entrada_decodificador, h, c, salidas_codificador)
            indice_predicho = salida.argmax(dim=1)
            palabra = vocab_salida_inverso.get(indice_predicho.item(), "<unk>")
            
            if palabra == "<eos>":
                break
            
            palabras_traducidas.append(palabra)
            entrada_decodificador = indice_predicho.unsqueeze(0)
            
    return " ".join(palabras_traducidas)


In [19]:
funcion_perdida = nn.CrossEntropyLoss(ignore_index=vocabulario_salida["<pad>"])
optimizador = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()), lr=0.01)

print("Iniciando entrenamiento del Seq2Seq con Atención...")
codificador.train()
decodificador.train()

for epoca in range(100):
    perdida_total = 0.0
    for i in range(len(tensor_entrada)):
        src = tensor_entrada[i].unsqueeze(0).to(dispositivo)
        trg = tensor_salida[i].unsqueeze(0).to(dispositivo)
        
        optimizador.zero_grad()
        salidas_codificador, (h, c) = codificador(src)
        
        perdida_oracion = 0.0
        entrada_decodificador = trg[:, 0].unsqueeze(1)  # Inicialización en <sos>
        
        # Decodificación y entrenamiento usando Teacher Forcing
        for t in range(1, trg.size(1)):
            token_objetivo = trg[:, t].item()
            if token_objetivo == vocabulario_salida["<pad>"]:
                continue
            salida, h, c = decodificador(entrada_decodificador, h, c, salidas_codificador)
            perdida_oracion += funcion_perdida(salida, trg[:, t])
            entrada_decodificador = trg[:, t].unsqueeze(1)  # Teacher forcing
            
        perdida_oracion.backward()
        optimizador.step()
        perdida_total += perdida_oracion.item()
        
    if (epoca + 1) % 10 == 0:
        print(f"Época [{epoca + 1}/100] | Pérdida total acumulada: {perdida_total:.4f}")


Iniciando entrenamiento del Seq2Seq con Atención...
Época [10/100] | Pérdida total acumulada: 0.5703
Época [20/100] | Pérdida total acumulada: 0.0286
Época [30/100] | Pérdida total acumulada: 0.0133
Época [40/100] | Pérdida total acumulada: 0.0091
Época [50/100] | Pérdida total acumulada: 0.0071
Época [60/100] | Pérdida total acumulada: 0.0058
Época [70/100] | Pérdida total acumulada: 0.0048
Época [80/100] | Pérdida total acumulada: 0.0042
Época [90/100] | Pérdida total acumulada: 0.0036
Época [100/100] | Pérdida total acumulada: 0.0032


In [ ]:
print("=== EJEMPLOS DE TRADUCCIÓN ===")

# 1. Traducir palabra suelta
palabra_test = "learning"
traduccion_palabra = traducir_oracion(palabra_test)
print(f"Inglés: {palabra_test} -> Francés: {traduccion_palabra}")

# 2. Traducir oración de entrenamiento completa
oracion_test = "I love deep learning"
traduccion_oracion = traducir_oracion(oracion_test)
print(f"Inglés: {oracion_test} -> Francés: {traduccion_oracion}")


=== EJEMPLOS DE TRADUCCIÓN ===
Inglés: learning -> Francés: j'adore l'apprentissage profond
Inglés: I love deep learning -> Francés: j'adore l'apprentissage profond
